# Notebook 04 — Data Integration
## Section 1: Geocode Stations (Multi-Month)

ICE data has `eva` IDs but **no coordinates**. Open-Meteo needs lat/lon.

**Steps**
1. Load `ice_cleaned` for Jul / Aug / Sep  
2. Collect unique stations  
3. Geocode with Nominatim (reuse existing coords if already saved)  
4. Save `station_coordinates.json`

**Next:** Section 2 — fetch hourly weather per station.

In [1]:
# =============================================================================
# Notebook 04 | Section 1: Geocode Stations (Multi-Month)
# =============================================================================
# Load ice_cleaned for all months → unique eva → Nominatim lat/lon.
# Reuses existing station_coordinates.json; only geocodes missing stations.
# =============================================================================

from __future__ import annotations

import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"

STATION_COORDS_PATH = REFERENCE_DIR / "station_coordinates.json"
GEOCODING_REPORT_PATH = REFERENCE_DIR / "geocoding_report_multi_month.json"

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "ICE-Train-Delay-Capstone/1.0 (university-ml-project)"
REQUEST_DELAY_SEC = 1.1  # Nominatim: max 1 request/sec


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]

# =============================================================================
# 1. LOAD ALL CLEANED MONTHS → UNIQUE STATIONS
# =============================================================================
frames = []
for month in TARGET_MONTHS:
    path = PROCESSED_DIR / f"ice_cleaned_{month}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nRun Notebook 03 first.")
    print(f"Loading: {path.name}")
    frames.append(pd.read_parquet(path, columns=["eva", "xml_station_name", "station_name", "id"]))

ice_df = pd.concat(frames, ignore_index=True)
print(f"Total rows across months: {len(ice_df):,}")

stations = (
    ice_df.groupby("eva")
    .agg(
        xml_station_name=("xml_station_name", lambda x: x.mode().iloc[0]),
        station_name=("station_name", lambda x: x.mode().iloc[0]),
        stop_count=("id", "count"),
    )
    .reset_index()
    .sort_values("stop_count", ascending=False)
)
print(f"Unique eva stations: {len(stations)}")
print()

# =============================================================================
# 2. LOAD EXISTING CACHE (reuse July coords if present)
# =============================================================================
cached_stations: dict = {}
if STATION_COORDS_PATH.exists():
    cached = load_json(STATION_COORDS_PATH)
    cached_stations = cached.get("stations", {})
    print(f"Loaded cache: {len(cached_stations)} stations from {STATION_COORDS_PATH.name}")
else:
    print("No existing coordinate cache — will geocode all stations.")
print()

# =============================================================================
# 3. GEOCODING HELPERS
# =============================================================================
def clean_station_name(name: str) -> str:
    name = re.sub(r"\s*\(S\).*", "", name)
    name = re.sub(r"\s*\(tief\).*", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def build_geocode_queries(xml_name: str, station_name: str) -> list[str]:
    queries = []
    for name in [xml_name, station_name, clean_station_name(xml_name)]:
        if name and f"{name}, Germany" not in queries:
            queries.append(f"{name}, Germany")
    return queries


def geocode_nominatim(query: str, timeout: int = 15) -> dict | None:
    headers = {"User-Agent": USER_AGENT}
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "de"}
    response = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=timeout)
    response.raise_for_status()
    results = response.json()
    if not results:
        return None
    hit = results[0]
    return {
        "latitude": float(hit["lat"]),
        "longitude": float(hit["lon"]),
        "display_name": hit.get("display_name", ""),
        "osm_class": hit.get("class", ""),
        "osm_type": hit.get("type", ""),
        "query_used": query,
    }


def geocode_station(eva: str, xml_name: str, station_name: str) -> dict:
    queries = build_geocode_queries(xml_name, station_name)
    for query in queries:
        try:
            result = geocode_nominatim(query)
            time.sleep(REQUEST_DELAY_SEC)
            if result is None:
                continue
            is_railway = (
                result["osm_class"] == "railway"
                or "station" in result["osm_type"].lower()
                or "bahnhof" in result["display_name"].lower()
            )
            return {
                "eva": eva,
                "xml_station_name": xml_name,
                "station_name": station_name,
                "status": "success",
                "latitude": result["latitude"],
                "longitude": result["longitude"],
                "display_name": result["display_name"],
                "osm_class": result["osm_class"],
                "osm_type": result["osm_type"],
                "query_used": result["query_used"],
                "is_railway_match": is_railway,
                "from_cache": False,
            }
        except Exception as exc:
            time.sleep(REQUEST_DELAY_SEC)
            return {
                "eva": eva,
                "xml_station_name": xml_name,
                "station_name": station_name,
                "status": "failed",
                "error": str(exc),
                "queries_tried": queries,
                "from_cache": False,
            }
    return {
        "eva": eva,
        "xml_station_name": xml_name,
        "station_name": station_name,
        "status": "failed",
        "error": "No results from Nominatim for any query variant",
        "queries_tried": queries,
        "from_cache": False,
    }


# =============================================================================
# 4. GEOCODE (SKIP IF CACHED)
# =============================================================================
geocoding_results: list[dict] = []
n_cached = 0
n_new = 0

to_geocode = []
for _, row in stations.iterrows():
    eva = str(row["eva"])
    if eva in cached_stations and "latitude" in cached_stations[eva]:
        c = cached_stations[eva]
        geocoding_results.append(
            {
                "eva": eva,
                "xml_station_name": row["xml_station_name"],
                "station_name": row["station_name"],
                "status": "success",
                "latitude": c["latitude"],
                "longitude": c["longitude"],
                "display_name": c.get("display_name", ""),
                "query_used": c.get("query_used", "cache"),
                "stop_count": int(row["stop_count"]),
                "from_cache": True,
            }
        )
        n_cached += 1
    else:
        to_geocode.append(row)

print(f"Already cached : {n_cached}")
print(f"Need to geocode: {len(to_geocode)}")
if to_geocode:
    print(f"(~{len(to_geocode) * REQUEST_DELAY_SEC / 60:.1f} min at 1 req/sec)")
print()

for i, row in enumerate(to_geocode, start=1):
    eva = str(row["eva"])
    xml_name = row["xml_station_name"]
    station_name = row["station_name"]
    print(f"  [{i}/{len(to_geocode)}] eva={eva}  {str(xml_name)[:40]}")
    result = geocode_station(eva, xml_name, station_name)
    result["stop_count"] = int(row["stop_count"])
    geocoding_results.append(result)
    n_new += 1
    if result["status"] == "success":
        print(
            f"         → ({result['latitude']:.4f}, {result['longitude']:.4f})  "
            f"[{result['query_used'][:50]}]"
        )
    else:
        print(f"         → FAILED: {result.get('error', 'unknown')}")

print()

# =============================================================================
# 5. SAVE LOOKUP
# =============================================================================
successful = [r for r in geocoding_results if r["status"] == "success"]
failed = [r for r in geocoding_results if r["status"] == "failed"]
success_rate = round(100 * len(successful) / max(len(geocoding_results), 1), 1)

station_coordinates = {
    r["eva"]: {
        "latitude": r["latitude"],
        "longitude": r["longitude"],
        "xml_station_name": r["xml_station_name"],
        "station_name": r.get("station_name", ""),
        "display_name": r.get("display_name", ""),
        "query_used": r.get("query_used", ""),
        "stop_count": r["stop_count"],
    }
    for r in successful
}

artifact = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "geocoding_service": "Nominatim (OpenStreetMap)",
        "geocoding_url": NOMINATIM_URL,
        "total_stations": len(stations),
        "successful": len(successful),
        "failed": len(failed),
        "from_cache": n_cached,
        "newly_geocoded": n_new,
        "success_rate_pct": success_rate,
        "primary_target": config["ml_tasks"]["primary"]["target"],
        "primary_metric": "mae",
    },
    "stations": station_coordinates,
    "failed_stations": failed,
}

with STATION_COORDS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(artifact), f, indent=2, ensure_ascii=False)

report = {
    "metadata": artifact["metadata"],
    "results": geocoding_results,
    "success_rate_pct": success_rate,
    "ready_for_section_2": success_rate >= 90.0,
}
with GEOCODING_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE")
print(sep)
print(f"Total stations  : {len(stations)}")
print(f"Geocoded OK     : {len(successful)}  ({success_rate}%)")
print(f"  from cache    : {n_cached}")
print(f"  newly geocoded: {n_new}")
print(f"Failed          : {len(failed)}")
print()
if successful:
    print("--- Top 5 by stop count ---")
    for s in sorted(successful, key=lambda x: x["stop_count"], reverse=True)[:5]:
        print(
            f"  eva={s['eva']}  {str(s['xml_station_name'])[:28]:<28} "
            f"→ ({s['latitude']:.4f}, {s['longitude']:.4f})"
        )
    print()
if failed:
    print("--- FAILED (review before Section 2) ---")
    for f_station in failed:
        print(f"  eva={f_station['eva']}  {f_station.get('xml_station_name', '')}")
    print()
print(f"Saved: {STATION_COORDS_PATH}")
print(f"Report: {GEOCODING_REPORT_PATH}")
print("Next: Section 2 — fetch hourly weather per station")
print(sep)

Loading: ice_cleaned_2024-07.parquet
Loading: ice_cleaned_2024-08.parquet
Loading: ice_cleaned_2024-09.parquet
Total rows across months: 418,257
Unique eva stations: 98

Loaded cache: 95 stations from station_coordinates.json

Already cached : 95
Need to geocode: 3
(~0.1 min at 1 req/sec)

  [1/3] eva=08000302  Plochingen
         → (48.7215, 9.4164)  [Plochingen, Germany]
  [2/3] eva=08000114  Fürth(Bay)Hbf
         → (49.4698, 10.9899)  [Fürth(Bay)Hbf, Germany]
  [3/3] eva=08011306  Berlin Friedrichstraße
         → (52.5202, 13.3870)  [Berlin Friedrichstraße, Germany]

Section 1 COMPLETE
Total stations  : 98
Geocoded OK     : 98  (100.0%)
  from cache    : 95
  newly geocoded: 3
Failed          : 0

--- Top 5 by stop count ---
  eva=08000105  Frankfurt(Main)Hbf           → (50.1067, 8.6626)
  eva=08000261  München Hbf                  → (48.1407, 11.5569)
  eva=08000152  Hannover Hbf                 → (52.3771, 9.7418)
  eva=08002549  Hamburg Hbf                  → (53.5532, 10.0064

# Notebook 04 — Data Integration
## Section 2: Fetch Hourly Weather (Multi-Month)

For each station with lat/lon, download Open-Meteo hourly weather for that month.

**Steps (per month)**
1. Load `station_coordinates.json` + date range from `ice_cleaned`
2. Call Open-Meteo Archive API per station
3. Save `weather_by_station_YYYY-MM.parquet` (skips if already exists)

**Next:** Section 3 — merge ICE + weather and save merged Parquet on disk.

In [2]:
# =============================================================================
# Notebook 04 | Section 2: Fetch Hourly Weather (Multi-Month)
# =============================================================================
# For each month: fetch Open-Meteo weather for all geocoded stations.
# SKIP_IF_EXISTS=True skips months that already have weather parquet.
# Per-station cache makes long runs resumable.
# =============================================================================

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
STATION_COORDS_PATH = REFERENCE_DIR / "station_coordinates.json"

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT = 60
REQUEST_DELAY_SEC = 0.3
TIMEZONE = "Europe/Berlin"
SKIP_IF_EXISTS = True  # set False to re-download

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def fetch_weather_for_station(
    eva: str,
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": TIMEZONE,
    }
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL, params=params, timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df["eva"] = eva
    df["latitude"] = data.get("latitude", latitude)
    df["longitude"] = data.get("longitude", longitude)
    df["timezone"] = data.get("timezone", TIMEZONE)
    return df


def process_month(target_month: str, stations: dict) -> dict:
    ice_path = PROCESSED_DIR / f"ice_cleaned_{target_month}.parquet"
    weather_output = PROCESSED_DIR / f"weather_by_station_{target_month}.parquet"
    weather_cache = PROCESSED_DIR / f"weather_by_station_{target_month}_cache.parquet"
    report_path = REFERENCE_DIR / f"weather_fetch_report_{target_month}.json"

    print("=" * 72)
    print(f"FETCH WEATHER — {target_month}")
    print("=" * 72)

    if not ice_path.exists():
        raise FileNotFoundError(f"Missing: {ice_path}")

    if SKIP_IF_EXISTS and weather_output.exists():
        wx = pd.read_parquet(weather_output)
        print(f"SKIP (already exists): {weather_output.name}")
        print(f"  rows={len(wx):,}  stations={wx['eva'].nunique()}")
        print()
        return {
            "month": target_month,
            "skipped": True,
            "total_rows": int(len(wx)),
            "unique_stations": int(wx["eva"].nunique()),
            "output_parquet": str(weather_output),
            "report": str(report_path) if report_path.exists() else None,
        }

    ice_df = pd.read_parquet(ice_path, columns=["time"])
    start_date = str(pd.to_datetime(ice_df["time"]).min().date())
    end_date = str(pd.to_datetime(ice_df["time"]).max().date())
    print(f"Date range : {start_date} → {end_date}")
    print(f"Stations   : {len(stations)}")
    print()

    already_fetched: set[str] = set()
    cached_frames: list[pd.DataFrame] = []
    if weather_cache.exists():
        cache_df = pd.read_parquet(weather_cache)
        already_fetched = set(cache_df["eva"].astype(str).unique())
        cached_frames.append(cache_df)
        print(f"Resuming from cache: {len(already_fetched)} stations")
        print()

    fetch_log: list[dict] = []
    new_frames: list[pd.DataFrame] = []
    stations_to_fetch = {
        eva: info for eva, info in stations.items() if str(eva) not in already_fetched
    }
    print(f"Fetching {len(stations_to_fetch)} stations...")
    print()

    for i, (eva, info) in enumerate(stations_to_fetch.items(), start=1):
        label = str(info.get("xml_station_name", eva))[:40]
        print(f"  [{i}/{len(stations_to_fetch)}] eva={eva}  {label}")
        try:
            wx_df = fetch_weather_for_station(
                eva=str(eva),
                latitude=info["latitude"],
                longitude=info["longitude"],
                start_date=start_date,
                end_date=end_date,
            )
            new_frames.append(wx_df)
            fetch_log.append(
                {"eva": eva, "status": "success", "rows": len(wx_df)}
            )
            print(f"         → {len(wx_df)} hourly rows")
            all_so_far = pd.concat(cached_frames + new_frames, ignore_index=True)
            all_so_far.to_parquet(weather_cache, index=False)
        except Exception as exc:
            fetch_log.append({"eva": eva, "status": "failed", "error": str(exc)})
            print(f"         → FAILED: {exc}")
        time.sleep(REQUEST_DELAY_SEC)

    print()
    all_frames = cached_frames + new_frames
    if not all_frames:
        raise ValueError(f"No weather data for {target_month}. Check internet/coords.")

    weather_all = pd.concat(all_frames, ignore_index=True)
    expected_cols = ["eva", "time", "latitude", "longitude"] + HOURLY_VARIABLES
    missing = [c for c in expected_cols if c not in weather_all.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    weather_all.to_parquet(weather_output, index=False)

    summary = {
        "total_rows": int(len(weather_all)),
        "unique_stations": int(weather_all["eva"].nunique()),
        "unique_hours": int(weather_all["time"].nunique()),
        "date_min": str(weather_all["time"].min()),
        "date_max": str(weather_all["time"].max()),
        "temperature_2m_mean": round(float(weather_all["temperature_2m"].mean()), 2),
        "precipitation_total_mm": round(float(weather_all["precipitation"].sum()), 2),
    }
    failed = [f for f in fetch_log if f["status"] == "failed"]

    report = {
        "metadata": {
            "notebook": "04_Data_Integration",
            "section": "Section 2",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "api_url": OPEN_METEO_ARCHIVE_URL,
            "date_range": {"start": start_date, "end": end_date},
            "timezone": TIMEZONE,
            "hourly_variables": HOURLY_VARIABLES,
            "primary_metric": "mae",
        },
        "weather_summary": summary,
        "fetch_log": fetch_log,
        "failed_count": len(failed),
        "output_parquet": str(weather_output),
        "ready_for_section_3": len(failed) == 0,
    }
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

    print(f"Saved: {weather_output}")
    print(
        f"Rows={summary['total_rows']:,}  "
        f"stations={summary['unique_stations']}  "
        f"mean temp={summary['temperature_2m_mean']}°C"
    )
    print()

    return {
        "month": target_month,
        "skipped": False,
        "total_rows": summary["total_rows"],
        "unique_stations": summary["unique_stations"],
        "failed": len(failed),
        "output_parquet": str(weather_output),
        "report": str(report_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]

if not STATION_COORDS_PATH.exists():
    raise FileNotFoundError(
        f"Missing: {STATION_COORDS_PATH}\nRun Notebook 04 Section 1 first."
    )

coords_data = load_json(STATION_COORDS_PATH)
stations = coords_data["stations"]

print("Notebook 04 | Section 2 — Multi-month weather fetch")
print(f"Months   : {', '.join(TARGET_MONTHS)}")
print(f"Stations : {len(stations)}")
print(f"SKIP_IF_EXISTS = {SKIP_IF_EXISTS}")
print()

month_results = [process_month(m, stations) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "weather_fetch_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "hourly_variables": HOURLY_VARIABLES,
    },
    "months": month_results,
    "totals": {
        "weather_rows_all": int(sum(r["total_rows"] for r in month_results)),
    },
}
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE")
print(sep)
print(f"{'Month':<10} {'Rows':>12} {'Stations':>10} {'Note':<12}")
for r in month_results:
    note = "skipped" if r.get("skipped") else "fetched"
    print(
        f"{r['month']:<10} {r['total_rows']:>12,} "
        f"{r['unique_stations']:>10} {note:<12}"
    )
print()
print(f"Summary: {summary_path}")
print("Next: Section 3 — merge ICE + weather → save ice_weather_merged_*.parquet")
print(sep)

Notebook 04 | Section 2 — Multi-month weather fetch
Months   : 2024-07, 2024-08, 2024-09
Stations : 98
SKIP_IF_EXISTS = True

FETCH WEATHER — 2024-07
SKIP (already exists): weather_by_station_2024-07.parquet
  rows=70,680  stations=95

FETCH WEATHER — 2024-08
Date range : 2024-08-01 → 2024-08-31
Stations   : 98

Fetching 98 stations...

  [1/98] eva=08000105  Frankfurt(Main)Hbf
         → 744 hourly rows
  [2/98] eva=08000261  München Hbf
         → 744 hourly rows
  [3/98] eva=08000152  Hannover Hbf
         → 744 hourly rows
  [4/98] eva=08002549  Hamburg Hbf
         → 744 hourly rows
  [5/98] eva=08010404  Berlin-Spandau
         → 744 hourly rows
  [6/98] eva=08000284  Nürnberg Hbf
         → 744 hourly rows
  [7/98] eva=08000207  Köln Hbf
         → 744 hourly rows
  [8/98] eva=08003200  Kassel-Wilhelmshöhe
         → 744 hourly rows
  [9/98] eva=08000128  Göttingen
         → 744 hourly rows
  [10/98] eva=08000085  Düsseldorf Hbf
         → 744 hourly rows
  [11/98] eva=08000115

# Notebook 04 — Data Integration
## Section 3: Merge ICE + Weather (Multi-Month)

**Join (left)**  
- Railway: `eva` + `departure_planned_hour_naive`  
- Weather: `eva` + hour (`weather_time`)

Keep all ICE rows. Weather can be null if no match.

**Per month we save on disk**  
`ice_weather_merged_YYYY-MM.parquet`

**Next:** Notebook 05 — EDA on the merged files.

In [3]:
# =============================================================================
# Notebook 04 | Section 3: Merge ICE + Weather (Multi-Month)
# =============================================================================
# Left-join ice_cleaned with weather_by_station on eva + hour.
# Saves ice_weather_merged_YYYY-MM.parquet to disk (required for later notebooks).
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"

WEATHER_COLS = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]
MERGE_RATE_THRESHOLD = 90.0  # % of mergeable rows that must get weather


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def process_month(target_month: str) -> dict:
    ice_path = PROCESSED_DIR / f"ice_cleaned_{target_month}.parquet"
    weather_path = PROCESSED_DIR / f"weather_by_station_{target_month}.parquet"
    merged_path = PROCESSED_DIR / f"ice_weather_merged_{target_month}.parquet"
    report_path = REFERENCE_DIR / f"merge_validation_report_{target_month}.json"

    print("=" * 72)
    print(f"MERGE ICE + WEATHER — {target_month}")
    print("=" * 72)

    if not ice_path.exists():
        raise FileNotFoundError(f"Missing: {ice_path}\nRun Notebook 03 first.")
    if not weather_path.exists():
        raise FileNotFoundError(f"Missing: {weather_path}\nRun Notebook 04 Section 2 first.")

    ice_df = pd.read_parquet(ice_path)
    weather_df = pd.read_parquet(weather_path)
    rows_before = len(ice_df)
    print(f"ICE rows     : {rows_before:,}")
    print(f"Weather rows : {len(weather_df):,}")

    # Ensure mergeable flag exists
    if "mergeable" not in ice_df.columns:
        ice_df["mergeable"] = ice_df["departure_planned_hour_naive"].notna()

    # Prepare weather join keys
    wx = weather_df.copy()
    wx["eva"] = wx["eva"].astype(str)
    ice_df["eva"] = ice_df["eva"].astype(str)

    wx["weather_time"] = pd.to_datetime(wx["time"]).dt.tz_localize(None)
    # Keep only keys + weather features (avoid duplicate lat/lon/time names)
    wx_keep = ["eva", "weather_time"] + [c for c in WEATHER_COLS if c in wx.columns]
    wx = wx[wx_keep].drop_duplicates(subset=["eva", "weather_time"], keep="first")

    ice_df["departure_planned_hour_naive"] = pd.to_datetime(
        ice_df["departure_planned_hour_naive"]
    )

    merged = ice_df.merge(
        wx,
        how="left",
        left_on=["eva", "departure_planned_hour_naive"],
        right_on=["eva", "weather_time"],
    )

    # weather matched if temperature (or first weather col) is not null
    primary_wx = WEATHER_COLS[0]
    merged["weather_matched"] = merged[primary_wx].notna()

    rows_after = len(merged)
    mergeable = merged["mergeable"]
    n_mergeable = int(mergeable.sum())
    n_matched = int((mergeable & merged["weather_matched"]).sum())
    n_non_mergeable = int((~mergeable).sum())
    overall_rate = round(100 * merged["weather_matched"].mean(), 2)
    mergeable_rate = round(100 * n_matched / n_mergeable, 2) if n_mergeable else 0.0

    print(f"Merged rows  : {rows_after:,}")
    print(f"Weather match (all rows)      : {overall_rate}%")
    print(f"Weather match (mergeable only): {mergeable_rate}%")
    print()

    # --- Validations ---
    validations = {
        "row_count_preserved": {
            "pass": rows_after == rows_before,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "rule": "Left join must not change row count",
        },
        "merge_rate_on_mergeable": {
            "pass": mergeable_rate >= MERGE_RATE_THRESHOLD,
            "merge_rate_pct": mergeable_rate,
            "threshold_pct": MERGE_RATE_THRESHOLD,
            "mergeable_rows": n_mergeable,
            "matched_rows": n_matched,
            "rule": f"≥ {MERGE_RATE_THRESHOLD}% of mergeable rows must have weather",
        },
        "no_duplicate_ids": {
            "pass": not merged["id"].duplicated().any(),
            "duplicate_count": int(merged["id"].duplicated().sum()),
            "rule": "id must be unique after merge",
        },
        "weather_columns_present": {
            "pass": all(c in merged.columns for c in WEATHER_COLS),
            "columns": WEATHER_COLS,
            "rule": "All weather variables must be in merged data",
        },
    }

    # Station gap check among mergeable rows
    if n_mergeable > 0:
        station_miss = (
            merged.loc[mergeable]
            .groupby("eva")["weather_matched"]
            .apply(lambda s: 1 - s.mean())
        )
        high_miss = int((station_miss > 0.5).sum())
    else:
        high_miss = 0
    validations["no_systematic_station_gaps"] = {
        "pass": high_miss == 0,
        "stations_with_high_miss_rate": high_miss,
        "rule": "No station should have >50% missing weather among mergeable rows",
    }

    failed = [k for k, v in validations.items() if v.get("pass") is False]
    if failed:
        raise ValueError(f"Merge validation failed for {target_month}: {failed}")

    # Save merged Parquet ON DISK
    merged.to_parquet(merged_path, index=False)
    print(f"SAVED ON DISK: {merged_path}")
    print(f"  size ≈ {merged_path.stat().st_size / 1e6:.1f} MB")
    print()

    delay_mean = round(float(merged["delay_in_min"].mean()), 2)
    temp_mean = (
        round(float(merged.loc[merged["weather_matched"], "temperature_2m"].mean()), 2)
        if merged["weather_matched"].any()
        else None
    )

    report = {
        "metadata": {
            "notebook": "04_Data_Integration",
            "section": "Section 3",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "join_type": "left",
            "join_keys": {
                "railway": ["eva", "departure_planned_hour_naive"],
                "weather": ["eva", "weather_time"],
            },
            "primary_target": "delay_in_min",
            "primary_metric": "mae",
            "merged_parquet_on_disk": str(merged_path),
        },
        "merge_summary": {
            "rows": rows_after,
            "columns": int(merged.shape[1]),
            "overall_merge_rate_pct": overall_rate,
            "mergeable_merge_rate_pct": mergeable_rate,
            "non_mergeable_rows": n_non_mergeable,
            "weather_matched_rows": int(merged["weather_matched"].sum()),
            "unique_eva_stations": int(merged["eva"].nunique()),
            "delay_mean_min": delay_mean,
            "weather_temperature_mean": temp_mean,
            "new_weather_columns": WEATHER_COLS,
        },
        "validations": validations,
        "validation_passed": len(failed) == 0,
        "output_parquet": str(merged_path),
        "ready_for_notebook_05": True,
    }

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

    print(f"Report: {report_path}")
    print(f"Mean delay: {delay_mean} min | Mean temp (matched): {temp_mean} °C")
    print()

    return {
        "month": target_month,
        "rows": rows_after,
        "mergeable_rate_pct": mergeable_rate,
        "overall_rate_pct": overall_rate,
        "delay_mean": delay_mean,
        "output_parquet": str(merged_path),
        "report": str(report_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]

print("Notebook 04 | Section 3 — Multi-month ICE + weather merge")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print("Output : ice_weather_merged_YYYY-MM.parquet (saved on disk)")
print()

month_results = [process_month(m) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "merge_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": "delay_in_min",
        "primary_metric": "mae",
    },
    "months": month_results,
    "totals": {
        "merged_rows_all": int(sum(r["rows"] for r in month_results)),
    },
    "merged_files_on_disk": [
        str(PROCESSED_DIR / f"ice_weather_merged_{m}.parquet") for m in TARGET_MONTHS
    ],
}
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 3 COMPLETE — merged data saved on disk")
print(sep)
print(f"{'Month':<10} {'Rows':>12} {'Match% (mergeable)':>20}")
for r in month_results:
    print(f"{r['month']:<10} {r['rows']:>12,} {r['mergeable_rate_pct']:>19.1f}%")
print()
print("Merged Parquets:")
for r in month_results:
    print(f"  {r['output_parquet']}")
print()
print(f"Summary: {summary_path}")
print("Next: Notebook 05 — EDA (load merged Parquet from disk)")
print(sep)

Notebook 04 | Section 3 — Multi-month ICE + weather merge
Months : 2024-07, 2024-08, 2024-09
Output : ice_weather_merged_YYYY-MM.parquet (saved on disk)

MERGE ICE + WEATHER — 2024-07
ICE rows     : 146,389
Weather rows : 70,680
Merged rows  : 146,389
Weather match (all rows)      : 90.35%
Weather match (mergeable only): 100.0%

SAVED ON DISK: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_weather_merged_2024-07.parquet
  size ≈ 6.3 MB

Report: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\merge_validation_report_2024-07.json
Mean delay: 10.92 min | Mean temp (matched): 20.29 °C

MERGE ICE + WEATHER — 2024-08
ICE rows     : 136,321
Weather rows : 72,912
Merged rows  : 136,321
Weather match (all rows)      : 90.08%
Weather match (mergeable only): 99.99%

SAVED ON DISK: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_weather_merged_2024-08.parquet
  size ≈ 5.8 MB

Report: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\referenc

# Notebook 04 — Data Integration
## Section 4: Integration Summary & Close-Out

Confirm all integration files exist on disk (especially merged Parquets).

**Ready for:** Notebook 05 — EDA on `ice_weather_merged_*.parquet`

In [4]:
# =============================================================================
# Notebook 04 | Section 4: Close-Out (Multi-Month)
# =============================================================================
# Verify coords, weather, merged Parquets on disk; save notebook_04_summary.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

print("Notebook 04 | Section 4 — Close-out")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    size = path.stat().st_size if ok else 0
    checklist.append(
        {
            "label": label,
            "path": str(path),
            "exists": ok,
            "size_mb": round(size / 1e6, 2) if ok else 0.0,
        }
    )
    status = "OK" if ok else "MISSING"
    size_txt = f"{size / 1e6:.1f} MB" if ok else "—"
    print(f"  [{status}] {label:<48} {size_txt}")
    return ok


print("=" * 72)
print("FILE CHECKLIST")
print("=" * 72)

all_ok = True
all_ok &= check("station_coordinates.json", REFERENCE_DIR / "station_coordinates.json")
all_ok &= check(
    "geocoding_report_multi_month.json",
    REFERENCE_DIR / "geocoding_report_multi_month.json",
)
print()

for month in TARGET_MONTHS:
    print(f"{month}")
    all_ok &= check(
        f"weather_by_station_{month}.parquet",
        PROCESSED_DIR / f"weather_by_station_{month}.parquet",
    )
    all_ok &= check(
        f"ice_weather_merged_{month}.parquet",
        PROCESSED_DIR / f"ice_weather_merged_{month}.parquet",
    )
    all_ok &= check(
        f"merge_validation_report_{month}.json",
        REFERENCE_DIR / f"merge_validation_report_{month}.json",
    )
    print()

all_ok &= check(
    "merge_summary_multi_month.json",
    REFERENCE_DIR / "merge_summary_multi_month.json",
)

merge_summary = load_json(REFERENCE_DIR / "merge_summary_multi_month.json")

print()
print("=" * 72)
print("MERGED DATA ON DISK (load with pd.read_parquet)")
print("=" * 72)
print(f"{'Month':<10} {'Rows':>12} {'Match%':>10} {'File'}")
month_detail = []
for m in merge_summary["months"]:
    path = Path(m["output_parquet"])
    n_disk = int(len(pd.read_parquet(path, columns=["id"]))) if path.exists() else None
    print(
        f"{m['month']:<10} {m['rows']:>12,} "
        f"{m['mergeable_rate_pct']:>9.1f}%  {path.name}"
    )
    if n_disk is not None and n_disk != m["rows"]:
        raise ValueError(f"Row mismatch {m['month']}: report={m['rows']}, disk={n_disk}")
    month_detail.append({**m, "rows_on_disk": n_disk})

print()
print(f"TOTAL merged rows: {merge_summary['totals']['merged_rows_all']:,}")
print()

print("=" * 72)
print("WHAT NOTEBOOK 04 DID")
print("=" * 72)
print("  1. Geocoded stations → lat/lon")
print("  2. Fetched hourly Open-Meteo weather (3 months)")
print("  3. Left-joined ICE + weather on eva + hour")
print("  4. Saved ice_weather_merged_*.parquet on disk")
print()

ready = bool(all_ok)
print("=" * 72)
print(f"Ready for Notebook 05 (EDA): {'YES' if ready else 'NO'}")
print("=" * 72)
if not ready:
    missing = [c["label"] for c in checklist if not c["exists"]]
    raise FileNotFoundError(f"Missing files: {missing}")

summary_path = REFERENCE_DIR / "notebook_04_summary.json"
summary = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 4",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "checklist": checklist,
    "all_files_ok": ready,
    "months": month_detail,
    "totals": merge_summary["totals"],
    "merged_data_location": str(PROCESSED_DIR),
    "merged_files": [f"ice_weather_merged_{m}.parquet" for m in TARGET_MONTHS],
    "next_notebook": {
        "id": "05",
        "goal": "EDA on merged data — delay distribution + weather vs delay (regression focus)",
        "inputs": [f"ice_weather_merged_{m}.parquet" for m in TARGET_MONTHS],
    },
    "ready_for_notebook_05": ready,
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print()
print(f"Saved: {summary_path}")
print("Next: Notebook 05 — Exploratory Data Analysis")

Notebook 04 | Section 4 — Close-out
Months : 2024-07, 2024-08, 2024-09

FILE CHECKLIST
  [OK] station_coordinates.json                         0.0 MB
  [OK] geocoding_report_multi_month.json                0.0 MB

2024-07
  [OK] weather_by_station_2024-07.parquet               0.3 MB
  [OK] ice_weather_merged_2024-07.parquet               6.3 MB
  [OK] merge_validation_report_2024-07.json             0.0 MB

2024-08
  [OK] weather_by_station_2024-08.parquet               0.3 MB
  [OK] ice_weather_merged_2024-08.parquet               5.8 MB
  [OK] merge_validation_report_2024-08.json             0.0 MB

2024-09
  [OK] weather_by_station_2024-09.parquet               0.3 MB
  [OK] ice_weather_merged_2024-09.parquet               5.8 MB
  [OK] merge_validation_report_2024-09.json             0.0 MB

  [OK] merge_summary_multi_month.json                   0.0 MB

MERGED DATA ON DISK (load with pd.read_parquet)
Month              Rows     Match% File
2024-07         146,389     100.0%  ice_